# LoRA + BERT 情感分析推理

将 waimai 训练的 LoRA 适配器加载到 bert-base-chinese 上，对单条文本进行情感分析，输出类别 0/1 及对应概率。

In [2]:
import os
import torch
from transformers import BertConfig, BertForSequenceClassification, BertTokenizer
from peft import PeftModel
from Bert_Config import CONFIG, setup_seed, build_tokenizer, generate_data, get_save_path

BERT_MODEL_PATH = "../bert-base-chinese"
LORA_ADAPTER_PATH = "./bert_lora_checkpoint/best"
MAX_LENGTH = CONFIG["MAX_LENGTH"]
NUM_CLASSES = CONFIG["NUM_CLASSES"]
DROPOUT = CONFIG["DROPOUT"]
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NAME = CONFIG["NAME"]
print(f"配置名称: {NAME}")
print(f"使用设备: {DEVICE}")
print(f"BERT 路径: {BERT_MODEL_PATH}")
print(f"LoRA 适配器路径: {LORA_ADAPTER_PATH}")

配置名称: shop
使用设备: cpu
BERT 路径: ../bert-base-chinese
LoRA 适配器路径: ./bert_lora_checkpoint/best


In [2]:
# 加载分词器
tokenizer = BertTokenizer.from_pretrained(BERT_MODEL_PATH)

# 加载基础 BERT 分类模型（与训练时配置一致）
config = BertConfig.from_pretrained(BERT_MODEL_PATH)
config.hidden_dropout_prob = DROPOUT
config.attention_probs_dropout_prob = DROPOUT
config.num_labels=NUM_CLASSES

base_model = BertForSequenceClassification.from_pretrained(BERT_MODEL_PATH, config=config)
model = PeftModel.from_pretrained(base_model, LORA_ADAPTER_PATH)
model = model.to(DEVICE)
model.eval()
print("✅ 已加载 BERT + LoRA 模型")

Some weights of the model checkpoint at ../bert-base-chinese were not used when initializing BertForSequenceClassification: ['cls.seq_relationship.bias', 'cls.predictions.transform.dense.weight', 'cls.predictions.bias', 'cls.predictions.transform.LayerNorm.bias', 'cls.seq_relationship.weight', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.dense.bias', 'cls.predictions.decoder.weight']
- This IS expected if you are initializing BertForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of BertForSequenceClassification were not initialized from the model checkpoint

✅ 已加载 BERT + LoRA 模型


In [3]:
def predict_sentiment(text: str):
    """
    对单条文本做情感分析。
    返回: (预测标签 0/1, 概率 P(0), 概率 P(1))
    """
    inputs = tokenizer(
        text,
        padding="max_length",
        max_length=MAX_LENGTH,
        truncation=True,
        return_tensors="pt",
    ).to(DEVICE)
    with torch.no_grad():
        logits = model(**inputs).logits
    probs = torch.softmax(logits, dim=1).squeeze(0).cpu().numpy()
    pred = int(logits.argmax(dim=1).item())
    return pred, float(probs[0]), float(probs[1])


def print_result(text: str, label: int, prob_0: float, prob_1: float):
    sentiment = "负面(0)" if label == 0 else "正面(1)"
    print(f"输入: {text}")
    print(f"情感: {sentiment}")
    print(f"P(0): {prob_0:.4f}")
    print(f"P(1): {prob_1:.4f}")

In [5]:
# 示例：输入一段话，输出情感分析结果（含 0/1 概率）
example_text = "肚子舒服"
label, prob_0, prob_1 = predict_sentiment(example_text)
print_result(example_text, label, prob_0, prob_1)

输入: 垃圾。
情感: 负面(0)
P(0): 0.9178
P(1): 0.0822


In [ ]:
# 可在此修改输入文本，重新运行本单元格即可
your_text = "味道很好，配送也快，下次还会点！"
label, prob_0, prob_1 = predict_sentiment(your_text)
print_result(your_text, label, prob_0, prob_1)